# EVALUACIÓN CONTINUA SEMANA 2
### **Nombre Completo:** Angela Jara  
### **Título de la Aplicación:** Algoritmo Genético para la Optimización de Turnos de Enfermería (Nurse Scheduling)

## 1) Explicación del problema con datos o estructura de la información

La asignación y programación de turnos de personal en el sector salud es un problema clásico de optimización combinatoria de alta complejidad ($NP$-hard). Diseñar un cronograma mensual o semanal libre de conflictos que respete tanto la salud del trabajador como la demanda asistencial del centro de salud es vital.

Para resolver este escenario real, modelamos una clínica con las siguientes características:
- **Personal:** Un equipo de **5 enfermeros/as**.
- **Periodo:** Una semana laboral completa de **7 días** (Lunes a Domingo).
- **Espacio de Búsqueda:** El número total de combinaciones posibles es $4^{35} \approx 1.18 \times 10^{21}$ configuraciones posibles. Un algoritmo de fuerza bruta es inviable.

## 2) Presentación de la población, genes y cromosoma

Para aplicar Algoritmos Genéticos, codificamos la información de la siguiente manera:

### A. El Gen
Cada celda representa el turno asignado a un enfermero específico en un día. Los turnos posibles se codifican con números enteros de la siguiente manera:
- **`0`**: Descanso (😴)
- **`1`**: Turno Mañana (☀️)
- **`2`**: Turno Tarde (🌆)
- **`3`**: Turno Noche (🌙)

### B. El Cromosoma (Individuo)
Representa un horario semanal completo para los 5 enfermeros. Se codifica como una lista plana de **35 genes** ($5 \text{ enfermeros} \times 7 \text{ días} = 35$ genes).
```
Índice 0 a 6:   Enfermero 1 (Lun a Dom)
Índice 7 a 13:  Enfermero 2 (Lun a Dom)
...
Índice 28 a 34: Enfermero 5 (Lun a Dom)
```

### C. La Población
Es un conjunto de cronogramas semanales individuales (población inicial generada al azar). En esta solución utilizaremos una población de **20 individuos**.

## 3) Presentación de la función fitness (función de evaluación)

Nuestra **función fitness** evalúa qué tan adecuado y seguro es un cronograma semanal propuesto. Se parte de una puntuación máxima de **100 puntos**, aplicando penalizaciones por el incumplimiento de restricciones:

### Restricciones Fuertes (Hard Constraints) - Penalización de -15 pts c/u
1. **Cobertura Diaria:** Todos los días de la semana debe haber al menos un enfermero asignado al turno Mañana (`1`), uno a la Tarde (`2`), y uno a la Noche (`3`).
2. **Descanso Post-Noche:** Ningún enfermero puede hacer un turno Mañana (`1`) inmediatamente al día siguiente de realizar un turno de Noche (`3`).

### Restricciones Suaves (Soft Constraints) - Penalización de -3 pts c/u
3. **Carga Laboral:** Ningún enfermero debe laborar más de 5 días en la semana.
4. **Días Seguidos:** Un enfermero no debe trabajar más de 5 días consecutivos sin gozar de al menos un día de descanso (`0`).

### Fórmula Matemática del Fitness
$$Fitness = \max\left(0, 100 - (15 \times \text{Violaciones\_Fuertes}) - (3 \times \text{Violaciones\_Suaves})\right)$$

## 4) Aplicar claramente cada paso del algoritmo

A continuación, se presenta la implementación modular bajo el enfoque de Programación Orientada a Objetos en Python.

In [1]:
import random

class NurseScheduleIndividual:
    def __init__(self, size=35, genes=None):
        self.size = size
        if genes is None:
            # Paso 1: Inicialización Aleatoria de los Turnos
            self.genes = [random.randint(0, 3) for _ in range(size)]
        else:
            self.genes = list(genes)
        self.fitness = self.calculate_fitness()

    def calculate_fitness(self):
        # Paso 2: Evaluación Fitness
        score = 100
        
        # Convertimos la lista de 35 genes en una matriz de 5 enfermeros por 7 días
        matrix = [self.genes[i*7:(i+1)*7] for i in range(5)]
        
        # ---- RESTRICCIONES FUERTES ----
        # 1. Cobertura diaria obligatoria
        for day in range(7):
            day_shifts = [matrix[nurse][day] for nurse in range(5)]
            # Verificar presencia de Mañana (1), Tarde (2) y Noche (3)
            for required_shift in [1, 2, 3]:
                if required_shift not in day_shifts:
                    score -= 15
                    
        # 2. Descanso post-noche
        for nurse in range(5):
            for day in range(6):
                if matrix[nurse][day] == 3 and matrix[nurse][day+1] == 1:
                    score -= 15

        # ---- RESTRICCIONES SUAVES ----
        for nurse in range(5):
            # 3. Máximo 5 días de trabajo por semana
            work_days = sum(1 for shift in matrix[nurse] if shift > 0)
            if work_days > 5:
                score -= 3
                
            # 4. Máximo de 5 días seguidos de trabajo sin descanso
            consecutive_work = 0
            max_consecutive = 0
            for day in range(7):
                if matrix[nurse][day] > 0:
                    consecutive_work += 1
                    if consecutive_work > max_consecutive:
                        max_consecutive = consecutive_work
                else:
                    consecutive_work = 0
            if max_consecutive > 5:
                score -= 3

        return max(0, score)

    def mutate(self):
        # Operador de Mutación
        pos = random.randint(0, self.size - 1)
        self.genes[pos] = random.randint(0, 3)
        self.fitness = self.calculate_fitness()

    def crossover(self, other):
        # Operador de Cruce (Single Point Crossover a nivel de semanas completas de enfermeros)
        cut_point = random.randint(1, 4) * 7  # El corte se hace alineado al fin de semana de un enfermero
        child_genes = self.genes[:cut_point] + other.genes[cut_point:]
        return NurseScheduleIndividual(genes=child_genes)

In [2]:
class HospitalGeneticAlgorithm:
    def __init__(self, population_size=40, mutation_rate=0.15):
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        # Población Inicial
        self.population = [NurseScheduleIndividual() for _ in range(population_size)]
        self.generation = 1

    def select_parent(self, total_fitness):
        # Selección mediante método de la Ruleta Proporcional
        pick = random.uniform(0, total_fitness)
        current = 0
        for ind in self.population:
            current += ind.fitness
            if current >= pick:
                return ind
        return random.choice(self.population)

    def evolve(self):
        new_population = []
        total_fitness = sum(ind.fitness for ind in self.population)
        if total_fitness == 0:
            # Reseteo en caso de que toda la población inicial sea fallida
            self.population = [NurseScheduleIndividual() for _ in range(self.population_size)]
            total_fitness = sum(ind.fitness for ind in self.population)
            
        for _ in range(self.population_size):
            parent1 = self.select_parent(total_fitness)
            parent2 = self.select_parent(total_fitness)
            
            child = parent1.crossover(parent2)
            
            if random.random() < self.mutation_rate:
                child.mutate()
                
            new_population.append(child)
            if child.fitness == 100:  # Solución óptima encontrada
                break
                
        return new_population

    def run(self, max_generations=5000):
        while self.generation <= max_generations:
            max_fit = max(ind.fitness for ind in self.population)
            best_individual = next(ind for ind in self.population if ind.fitness == max_fit)
            
            if self.generation % 100 == 0 or max_fit == 100:
                print(f"Generación {self.generation} | Mejor Fitness = {max_fit}")
            
            if max_fit == 100:  # Criterio de Parada
                print(f"\n¡Solución óptima sin colisiones encontrada en la generación {self.generation}!")
                return best_individual
                
            self.population = self.evolve()
            self.generation += 1
            
        print("\nSe alcanzó el límite de generaciones. Retornando mejor solución obtenida.")
        return max(self.population, key=lambda x: x.fitness)

## 5) Presentación de la solución generada y explicación en el contexto del problema

In [3]:
# Instanciar y ejecutar el Algoritmo Genético
ga = HospitalGeneticAlgorithm(population_size=100, mutation_rate=0.2)
optimal_schedule = ga.run()

Generación 100 | Mejor Fitness = 97
Generación 193 | Mejor Fitness = 100

¡Solución óptima sin colisiones encontrada en la generación 193!


In [4]:
def display_schedule(individual):
    days = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
    shift_emojis = {0: "😴 Descanso", 1: "☀️ Mañana  ", 2: "🌆 Tarde   ", 3: "🌙 Noche   "}
    matrix = [individual.genes[i*7:(i+1)*7] for i in range(5)]
    
    print("=" * 85)
    print(f"                 CRONOGRAMA DE TURNOS OPTIMIZADO (Fitness: {individual.fitness}/100)")
    print("=" * 85)
    
    header = "| Enfermero   | " + " | ".join(f"{day:<9}" for day in days) + " |"
    print(header)
    print("-" * len(header))
    
    for i in range(5):
        row_str = f"| Enfermero {i+1} | "
        row_str += " | ".join(f"{shift_emojis[matrix[i][day]]}" for day in range(7))
        row_str += " |"
        print(row_str)
        
    print("=" * 85)
    
    print("\nAnálisis del Cumplimiento de Restricciones:")
    print("✅ Cobertura diaria (Mañana, Tarde, Noche garantizados todos los días).")
    print("✅ Seguridad y Salud Laboral: No hay turnos Mañana consecutivos al turno Noche.")
    print("✅ Confort laboral: Ningún enfermero trabaja más de 5 días seguidos o supera 5 días semanales.")

display_schedule(optimal_schedule)

                 CRONOGRAMA DE TURNOS OPTIMIZADO (Fitness: 100/100)
| Enfermero   | Lunes     | Martes    | Miércoles | Jueves    | Viernes   | Sábado    | Domingo   |
---------------------------------------------------------------------------------------------------
| Enfermero 1 | ☀️ Mañana   | ☀️ Mañana   | 😴 Descanso | 🌆 Tarde    | ☀️ Mañana   | 😴 Descanso | 🌆 Tarde    |
| Enfermero 2 | 🌙 Noche    | 🌙 Noche    | 🌙 Noche    | 🌙 Noche    | 😴 Descanso | 😴 Descanso | 🌙 Noche    |
| Enfermero 3 | 🌆 Tarde    | 😴 Descanso | ☀️ Mañana   | 😴 Descanso | 🌆 Tarde    | 🌆 Tarde    | ☀️ Mañana   |
| Enfermero 4 | 😴 Descanso | 😴 Descanso | 🌆 Tarde    | ☀️ Mañana   | 🌙 Noche    | 🌙 Noche    | 🌆 Tarde    |
| Enfermero 5 | 🌆 Tarde    | 🌆 Tarde    | 🌆 Tarde    | 😴 Descanso | 😴 Descanso | ☀️ Mañana   | 🌙 Noche    |

Análisis del Cumplimiento de Restricciones:
✅ Cobertura diaria (Mañana, Tarde, Noche garantizados todos los días).
✅ Seguridad y Salud Laboral: No hay turnos Mañana consecutivos al turno No